# CLOY Processing

This notebook is for you to work with the CLOY data folder.

**Author:** Kelly Chen\
**Date:** 9/29/26

**Table of Contents:**
1. [Goals and Tasks](#sec1)
2. [Vector of Words](#sec2)
3. [Cosine Similarity](#sec3)
4. [Top 5 Most Similar Threads](#sec4)
5. [Verification](#sec5)

<a id="sec1"></a>

## 1. Goals and Tasks

There are two main steps:

1. Turn every CLOY thread (a nested structure of the main post and subsequent comments) into a vector of words (as shown in slides 23-24 of Day 11 notes). [Spend some time looking at the structure of a single thread, since it is nested!]
2. Calculate the similarity score between every two threads  (with cosine similarity) [See sample code below], and save the dataframe into a CSV file.

Additionally:
1. Write a function that given one thread ID, prints out the threadID and subject title of the thread for the top 5 most similar threads.
2. Verification: Use the thread_explorer app to verify that these threads are indeed similar (you can search for the threads and read their content)
3. Remove stop words - Do you think stop words are affecting your results?  What happens if you get rid of them? [TextBlob has a list of stop words.]

<a id="sec2"></a>

## 2. Vector of Words

Import libraries:

In [241]:
import json
from textblob import TextBlob
from textblob import Word
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer #for vector
from sklearn.metrics.pairwise import cosine_similarity 

import string

Print threads by reading jsonl file:

In [ ]:
threads = []

file_path = "data/cs315-week4-data/RedditExplorer/cloy_chronological.jsonl"

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip(): 
            data = json.loads(line)
            threads.append(data) 

print(threads[1]) #read first thread to check

{'post_id': 'ffh52z', 'timestamp': 1583693959, 'author': 'bruisedlee123', 'subreddit': 'CrashLandingOnYou', 'title': 'ok, let’s get this started! What are your thoughts about the show?', 'selftext': 'I haven’t seen a kdrama for the longest time and it was actually my dad who chose to stream CLOY on Netflix. He stopped watching after the second episode but I went on after immediately falling in love with the leads. Hyun Bin is just so charismatic as Capt. Ri Jeong Hyeok and Son Ye-jin is perfect as Yoon Se-ri. She’s definitely got the rom-com and drama genre down pat! I also loved Capt. Ri’s squad, most especially Pyo Chi Su ☺️ (we can probably talk about individual characters next time).\n\nCLOY tackled politics, drama and deception, family issues, friendships - and best of all, pure unconditional love, love brought about by fate - all in one highly entertaining and well-written package. \n\nI’m planning on doing a rewatch soon, when I get a chance!', 'url': 'https://www.reddit.com/r/C

Get comments:

In [81]:
comments = []
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip(): 
            data = json.loads(line)
            comments.append(data["comments"]) 

print(comments[1]) #check comments for first thread

[{'comment_id': 'fk0rooq', 'parent_id': 't3_ffh52z', 'author': 'chimichangarolls', 'comment_body': "i'm so emotionally drained and exhausted from being in awe of this show even though i finished watching 2 weeks ago. my brain doesn't work cuz i'm thinking about CLOY and my favorite part of it other than the couple is that the women have really 3D roles... like Seri can cry nonstop but you still know she a boss.", 'score': 19, 'created_utc': 1583764623, 'replies': [{'comment_id': 'fk1c27d', 'parent_id': 't1_fk0rooq', 'author': 'bruisedlee123', 'comment_body': 'Did you have a favorite episode? Mine would have to be ep. 9 ☺️', 'score': 2, 'created_utc': 1583776621, 'replies': [{'comment_id': 'fsfm0lq', 'parent_id': 't1_fk1c27d', 'author': 'tmstms', 'comment_body': "I have not finished it yet, but I already see your point. Episode 9 is when you feel things are going to end happily. Also the stuff in RJH;s parents' home is priceless.", 'score': 2, 'created_utc': 1590941768}]}]}, {'comment_i

This code does not get the comments text for a reply to a main thread. Instead, write a (recursive) function that will get only the main thread comments text and any reply comments. 

In [82]:
def get_comments(comments):
    text = []

    for comment in comments:
        if "comment_body" in comment:
            text.append(comment["comment_body"]) # add this comment's text

        if "replies" in comment:
            text.extend(get_comments(comment["replies"])) # if replies add to comment 

    return text

In [83]:
comments_text = get_comments(comments[1])
print(comments_text) #gets all comments and replies; wrong order however

["i'm so emotionally drained and exhausted from being in awe of this show even though i finished watching 2 weeks ago. my brain doesn't work cuz i'm thinking about CLOY and my favorite part of it other than the couple is that the women have really 3D roles... like Seri can cry nonstop but you still know she a boss.", 'Did you have a favorite episode? Mine would have to be ep. 9 ☺️', "I have not finished it yet, but I already see your point. Episode 9 is when you feel things are going to end happily. Also the stuff in RJH;s parents' home is priceless.", 'I love this show. I stumbled across it on Netflix, gave it a try, and fell in love. I finished it this morning and already miss it. I love Yoon Se-Ri and Jeong Hyeok and nearly all of the other characters. I was so happy with the ending. I’ve added two more kdramas to my Netflix list.', 'Which ones!', 'Itaewon Class, Chocolate, When the Camellia Blooms', 'Thank you :)', "I'm currently obsessed with Hyun Bin. I've never watched Korean dr

Once I have the comments and replies for a thread, I can combine it together with the text from the main thread post.

I will first write a function that gets the text of the thread post written in them and combine it with the comments and replies.

In [138]:
def get_thread_text(thread):
    text = [thread.get("selftext", "")]

    # add comments and replies to main thread text
    text.extend(get_comments(thread.get("comments", [])))
    
    # remove punctuation
    text = [
        "".join(ch for ch in sentence if ch not in string.punctuation)
        for sentence in text
    ]

    return " ".join(text)

Check if the combine function works using the first thread as test:

In [146]:
print(get_thread_text(threads[1]))

I haven’t seen a kdrama for the longest time and it was actually my dad who chose to stream CLOY on Netflix He stopped watching after the second episode but I went on after immediately falling in love with the leads Hyun Bin is just so charismatic as Capt Ri Jeong Hyeok and Son Yejin is perfect as Yoon Seri She’s definitely got the romcom and drama genre down pat I also loved Capt Ri’s squad most especially Pyo Chi Su ☺️ we can probably talk about individual characters next time

CLOY tackled politics drama and deception family issues friendships  and best of all pure unconditional love love brought about by fate  all in one highly entertaining and wellwritten package 

I’m planning on doing a rewatch soon when I get a chance im so emotionally drained and exhausted from being in awe of this show even though i finished watching 2 weeks ago my brain doesnt work cuz im thinking about CLOY and my favorite part of it other than the couple is that the women have really 3D roles like Seri can

Yay, it works! Now compile results of all text into a df for vectorizing:

In [147]:
result = []

for thread in threads: 
    result.append({
        "threadID": thread["post_id"],
        "title": thread["title"], #add title of thread post
        "text": get_thread_text(thread) # all text 
    })

df = pd.DataFrame(result)

In [148]:
df.head()

,threadID,title,text
0,dkxewd,Crash Landing On You has been created,deleted
1,ffh52z,"ok, let’s get this started! What are your thou...",I haven’t seen a kdrama for the longest time a...
2,fguorr,Find someone who’ll make you noodles just like...,😘❤️ Man those noodles looked so good that I w...
3,fhcrpl,Captain Ri’s squad! Who’s your favorite?,I cant narrow it down to just one I love them...
4,fj5byj,Why I think we all need a second season of CLO...,I binged crash landing on you this week and ca...


#### Vectorize into bag-of-words model

Create the vectorizer:

In [149]:
vectorizer = CountVectorizer()

Fit to the text:

In [150]:
X = vectorizer.fit_transform(df["text"])

In [151]:
print(X.shape) #rows, cols or num of threads, num of unique words

(3075, 34128)


There are 3075 threads and 34128 unique words.

View the words:

In [152]:
words = vectorizer.get_feature_names_out()
print(words)

['00000001' '00001' '0001' ... '희망도' 'ﾟヮﾟ' '𝓷𝓲𝓬𝓮']


There appears to be numbers mixed in with words. Words also include Korean characters and some words in other special fonts.

Turn bag of words into df:

In [ ]:
words_df = pd.DataFrame(
    X.toarray(),
    columns=words
)

words_df["threadID"] = df["threadID"].values # add thread/post id as new col

print(words_df.head())

   00000001  00001  0001  0010  0011  0030  0038  007  01  014  ...  화이팅  흐음  \
0         0      0     0     0     0     0     0    0   0    0  ...    0   0   
1         0      0     0     0     0     0     0    0   0    0  ...    0   0   
2         0      0     0     0     0     0     0    0   0    0  ...    0   0   
3         0      0     0     0     0     0     0    0   0    0  ...    0   0   
4         0      0     0     0     0     0     0    0   0    0  ...    0   0   

   흔들렸던  흥부와  흥부전  흩날리던  희망도  ﾟヮﾟ  𝓷𝓲𝓬𝓮  threadID  
0     0    0    0     0    0    0     0    dkxewd  
1     0    0    0     0    0    0     0    ffh52z  
2     0    0    0     0    0    0     0    fguorr  
3     0    0    0     0    0    0     0    fhcrpl  
4     0    0    0     0    0    0     0    fj5byj  

[5 rows x 34129 columns]


<a id="sec3"></a>

## 3. Cosine Similarity

Now I need to calculate the cosine similarity score between every two threads and save the resulting df into a csv file.

In [185]:
similarity = cosine_similarity(X) #matrix of scores

Turn the matrix into a df:

In [186]:
similarity_df = pd.DataFrame(
    similarity,
    index=df["threadID"],
    columns=df["threadID"]
)

similarity_df.head()

threadID,dkxewd,ffh52z,fguorr,fhcrpl,fj5byj,fjimic,fjkurx,fkuhp7,flxz1f,fmbrra,...,1s0qkdq,1s10elb,1s10kr2,1s10qcn,1s13t5y,1s637j8,1s7yg7f,1s8aeiv,1s8i9nn,1s8ju38
threadID,,,,,,,,,,,,,,,,,,,,,
dkxewd,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ffh52z,0.0,1.000000,0.524368,0.763944,0.744212,0.263959,0.753940,0.820400,0.566837,0.457172,...,0.809015,0.504504,0.0,0.515918,0.611823,0.762717,0.796795,0.632393,0.288340,0.219380
fguorr,0.0,0.524368,1.000000,0.492458,0.433476,0.169492,0.385794,0.458279,0.376068,0.172516,...,0.534866,0.436004,0.0,0.408815,0.420985,0.470984,0.534235,0.436839,0.139389,0.140488
fhcrpl,0.0,0.763944,0.492458,1.000000,0.636442,0.217522,0.647365,0.714295,0.513503,0.404018,...,0.721160,0.426701,0.0,0.433259,0.521255,0.686258,0.732883,0.561959,0.280293,0.199866
fj5byj,0.0,0.744212,0.433476,0.636442,1.000000,0.192500,0.601375,0.768876,0.536834,0.424743,...,0.795845,0.501670,0.0,0.532209,0.554845,0.701672,0.776361,0.487740,0.375554,0.276453


The function calculates all the pairwise cosine similarity scores (for every pair). Cosine similarity scores are from 0 (no similarity) to 1 (very similar/same).

Check similarity between threads 1 and 3:

In [187]:
similarity[0][2]

np.float64(0.0)

For 2 and 4:

In [188]:
similarity[1][3]

np.float64(0.7639444745604439)

Save df as CSV file:

In [189]:
similarity_df.to_csv("cloy_thread_cosine_similarity.csv")

<a id="sec4"></a>

## 4. Top 5 Most Similar Threads

This function will take in a threadID and return the threadID and subject title for the top 5 most similar threads to that. This builds off of the cosine similarity task above. 

Start with selecting a random threadID: 

In [191]:
thread_id = df["threadID"].iloc[7] #index with iloc since threadID should be first col

print(thread_id)

fkuhp7


Once I have a thread ID (using first one as example), I was use it as my given thread to find the top 5 most similar threads to it.

In [192]:
similarities = similarity_df.loc[thread_id] #loc this time to access by name

print(similarities) # prints cosine similarity scores for that thread

threadID
dkxewd     0.000000
ffh52z     0.820400
fguorr     0.458279
fhcrpl     0.714295
fj5byj     0.768876
             ...   
1s637j8    0.749615
1s7yg7f    0.810731
1s8aeiv    0.566696
1s8i9nn    0.328671
1s8ju38    0.246169
Name: fkuhp7, Length: 3075, dtype: float64


Because the first row is itself, we need to remove it:

In [193]:
similarities = similarities.drop(thread_id)

Now sort the scores by most similar (descending):

In [194]:
similarities = similarities.sort_values(ascending=False)
top5 = similarities.head(5) #get top 5
top5

threadID
17tpx8l    0.863617
u1lawc     0.853143
gypswi     0.846279
lln7rk     0.846187
i42ez5     0.843166
Name: fkuhp7, dtype: float64

Print the thread ID and subject titles for the top 5:

In [197]:
for id, score in top5.items():
    title = df.loc[df["threadID"] == id, "title"].iloc[0]

    print(id, "||", title, "|| similarity:", round(score, 3)) #print similarity too!

17tpx8l || I just finished “Crash Landing On You” and I will never be the same. || similarity: 0.864
u1lawc || Just finished the show… || similarity: 0.853
gypswi || How has watching CLOY made an impact in your life? || similarity: 0.846
lln7rk || Depression after finishing this show? anyone else? || similarity: 0.846
i42ez5 || How come you started watching it? || similarity: 0.843


Now that I have all the steps, I can put everything together into a function:

In [201]:
def top5_similar_threads(thread_id):

    similarities = similarity_df.loc[thread_id] #get thread

    similarities = similarities.drop(thread_id) #drop similarity for itself

    similarities = similarities.sort_values(ascending=False) #sort 

    top5 = similarities.head(5) # get top 5

    print("ThreadID:", thread_id)
    print(f"\nTop 5 most similar threads to: {thread_id}") 

    for id, score in top5.items():
        title = df.loc[df["threadID"] == id, "title"].iloc[0]

        print(id, "||", title, "|| similarity:", round(score, 3))

Test with the example I used earlier:

In [202]:
top5_similar_threads(df["threadID"].iloc[7])

ThreadID: fkuhp7

Top 5 most similar threads to: fkuhp7
17tpx8l || I just finished “Crash Landing On You” and I will never be the same. || similarity: 0.864
u1lawc || Just finished the show… || similarity: 0.853
gypswi || How has watching CLOY made an impact in your life? || similarity: 0.846
lln7rk || Depression after finishing this show? anyone else? || similarity: 0.846
i42ez5 || How come you started watching it? || similarity: 0.843


<a id="sec5"></a>

## 5. Verification

I opened the thread_explorer app and verified that the threads text had similarities among them. This means the cosine similarity captured some overlap in topics, although some seemed to be based off of common words as well.

<a id="sec6"></a>

## 6. Stop Words

There are definitely many stop words in these thread texts, and there will likely be impacts if I remove them. Let's view what they are first:

In [208]:
import nltk
nltk.download("stopwords") #download stopwords

from nltk.corpus import stopwords

stopwords = stopwords.words("english")

print(stopwords)

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

[nltk_data] Downloading package stopwords to /Users/Kelly/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Now vectorize into another bag of words without these stopwords:

In [209]:
vectorizer_no_stop = CountVectorizer(stop_words=stopwords)

X_no_stop = vectorizer_no_stop.fit_transform(df["text"])

Calculate cosine similarity again and turn into df:

In [226]:
similarity_no_stop = cosine_similarity(X_no_stop)
df_no_stop = pd.DataFrame(
    similarity_no_stop,
    index=df["threadID"],
    columns=df["threadID"]
)

In [228]:
df_no_stop.head()

threadID,dkxewd,ffh52z,fguorr,fhcrpl,fj5byj,fjimic,fjkurx,fkuhp7,flxz1f,fmbrra,...,1s0qkdq,1s10elb,1s10kr2,1s10qcn,1s13t5y,1s637j8,1s7yg7f,1s8aeiv,1s8i9nn,1s8ju38
threadID,,,,,,,,,,,,,,,,,,,,,
dkxewd,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ffh52z,0.0,1.000000,0.076595,0.364966,0.417704,0.103018,0.395581,0.485486,0.135826,0.141958,...,0.441616,0.201559,0.0,0.225409,0.221681,0.359654,0.375473,0.216408,0.123817,0.086738
fguorr,0.0,0.076595,1.000000,0.185864,0.056769,0.032444,0.009228,0.077784,0.012369,0.030657,...,0.153361,0.062954,0.0,0.055317,0.070799,0.042859,0.107096,0.027037,0.000000,0.000000
fhcrpl,0.0,0.364966,0.185864,1.000000,0.233753,0.036181,0.205829,0.306153,0.101156,0.113961,...,0.344867,0.102968,0.0,0.123377,0.122818,0.245804,0.294251,0.110554,0.091115,0.081236
fj5byj,0.0,0.417704,0.056769,0.233753,1.000000,0.059222,0.169982,0.378873,0.125206,0.137355,...,0.461535,0.150432,0.0,0.167064,0.107694,0.384046,0.419562,0.127866,0.195974,0.103612


Compare top 5 threads now with and without stopwords using this function:

In [ ]:
def compare_top5(thread_id):
    with_stop = (
        similarity_df.loc[thread_id]
        .drop(thread_id)
        .sort_values(ascending=False)
        .head(5)
        .reset_index()
    ) #combines the top5 function

    no_stop = (
        df_no_stop.loc[thread_id]
        .drop(thread_id)
        .sort_values(ascending=False)
        .head(5)
        .reset_index()
    )

    comparison = pd.concat([with_stop, no_stop], axis=1)

    print(comparison)

In [240]:
compare_top5(df["threadID"].iloc[7])

  threadID    fkuhp7 threadID    fkuhp7
0  17tpx8l  0.863617  1bmn3p0  0.573273
1   u1lawc  0.853143   xlso1s  0.560480
2   gypswi  0.846279   u1lawc  0.555265
3   lln7rk  0.846187  1bh43sz  0.542233
4   i42ez5  0.843166  1i7qtqq  0.537537


There is a decrease in the cosine similarity scores from about 0.8 to 0.5 for the top 5 similar dramas to thread with ID = fkuhp7.